# Bernoulli Mixture Models for Binarised MNIST

This notebook implements a **Bernoulli mixture model** from scratch and trains it with the
**Expectation-Maximisation (EM)** algorithm. The aim is to model binarised MNIST images as
samples generated by latent mixture components.

The project demonstrates:

- data preprocessing for MNIST;
- E-step and M-step implementation in NumPy;
- negative log-likelihood monitoring;
- experiments with model capacity, sample size and random initialisation;
- sampling new digit-like images from the fitted model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

np.random.seed(42)


## Utility functions

Each image is represented as a binary vector $x \in \{0,1\}^{784}$. Each mixture component has
a vector of Bernoulli probabilities, one for each pixel.


In [ ]:
def plot_mnist_grid(images, titles=None, n_cols=5, figsize=(10, 4)):
    """Plot a collection of flattened 28x28 MNIST images."""
    images = np.asarray(images)
    n_images = len(images)
    n_rows = int(np.ceil(n_images / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = np.asarray(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i, image in enumerate(images):
        axes[i].imshow(image.reshape(28, 28), cmap="gray", interpolation="nearest")
        if titles is not None:
            axes[i].set_title(str(titles[i]), fontsize=10)

    plt.tight_layout()
    plt.show()


def logsumexp(a, axis=None, keepdims=False):
    """Numerically stable log-sum-exp implementation."""
    a = np.asarray(a)
    a_max = np.max(a, axis=axis, keepdims=True)
    out = a_max + np.log(np.sum(np.exp(a - a_max), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)
    return out


def bernoulli_mixture_nll(pi, mu, X, eps=1e-6):
    """Negative log-likelihood of a Bernoulli mixture model."""
    mu = np.clip(mu, eps, 1 - eps)
    pi = np.clip(pi, eps, 1.0)

    log_mu = np.log(mu)
    log_1m = np.log(1 - mu)
    log_pi = np.log(pi)

    log_likelihood = X @ log_mu.T + (1 - X) @ log_1m.T
    log_px = logsumexp(log_likelihood + log_pi[None, :], axis=1)
    return -np.sum(log_px)


## Load and binarise MNIST

To make the mixture structure visible, I restrict the dataset to three visually different digits.
The notebook can be re-run with a different set of digits.


In [ ]:
def load_binarised_mnist(digits=(1, 8, 0), threshold=0.5):
    """Load selected MNIST digits and convert greyscale pixels into binary values."""
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)

    X_image = mnist.data.astype(np.float32) / 255.0
    y = mnist.target.astype(int)

    mask = np.isin(y, digits)
    X_image = X_image[mask]
    y = y[mask]

    X_binary = (X_image > threshold).astype(np.int8)
    return X_binary, y


X, y = load_binarised_mnist(digits=(1, 8, 0))
print(f"Dataset shape: {X.shape}")


In [ ]:
plot_mnist_grid(X[:10], titles=y[:10], n_cols=5, figsize=(8, 4))


## EM algorithm

The latent variable $z_n$ indicates which component generated image $x_n$.

- **E-step:** compute responsibilities $\gamma_{nk}=p(z_n=k \mid x_n)$.
- **M-step:** update the mixture weights and pixel probabilities using the expected component assignments.

The responsibility calculation is done in log-space using a numerically stable log-sum-exp operation. This matters because each image has 784 pixels, so directly multiplying Bernoulli probabilities can underflow to zero.

In [ ]:
def e_step(pi, mu, X, eps=1e-6):
    """Compute component responsibilities for each observation."""
    mu = np.clip(mu, eps, 1 - eps)
    pi = np.clip(pi, eps, 1.0)

    log_mu = np.log(mu)
    log_1m = np.log(1 - mu)
    log_pi = np.log(pi)

    log_joint = X @ log_mu.T + (1 - X) @ log_1m.T + log_pi[None, :]
    log_norm = logsumexp(log_joint, axis=1, keepdims=True)
    return np.exp(log_joint - log_norm)


def m_step(gamma, X, eps=1e-6):
    """Update mixture weights and Bernoulli parameters."""
    n_samples = X.shape[0]
    effective_counts = gamma.sum(axis=0)

    pi = effective_counts / n_samples
    mu = (gamma.T @ X) / effective_counts[:, None]
    mu = np.clip(mu, eps, 1 - eps)
    return pi, mu


def fit_bernoulli_mixture(X, n_components=3, n_iter=20, seed=0):
    """Fit a Bernoulli mixture model with EM."""
    rng = np.random.default_rng(seed)
    _, n_features = X.shape

    pi = np.ones(n_components) / n_components
    mu = rng.uniform(0.25, 0.75, size=(n_components, n_features))

    losses = []
    for _ in range(n_iter):
        gamma = e_step(pi, mu, X)
        pi, mu = m_step(gamma, X)
        losses.append(bernoulli_mixture_nll(pi, mu, X))

    return {"pi": pi, "mu": mu, "gamma": gamma, "losses": np.array(losses)}


## Baseline fit

I first fit a three-component model, matching the number of selected digit classes. The component
means can be interpreted as pixel-wise probabilities of being active under each latent cluster.


In [ ]:
n_train = 600
n_components = 3
n_iter = 20

result = fit_bernoulli_mixture(X[:n_train], n_components=n_components, n_iter=n_iter, seed=0)

print("Final mixture weights:", np.round(result["pi"], 3))
print("Final negative log-likelihood:", round(result["losses"][-1], 2))


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, n_iter + 1), result["losses"], marker="o")
plt.xlabel("EM iteration")
plt.ylabel("Negative log-likelihood")
plt.title("EM training curve")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()


In [ ]:
plot_mnist_grid(
    result["mu"],
    titles=[f"component {k}" for k in range(n_components)],
    n_cols=3,
    figsize=(6, 2.5),
)


## Model comparison experiments

The next cells vary the number of components, the number of training examples and the random seed.
These checks are useful because EM is sensitive to both model capacity and initialisation.


In [ ]:
n_iter = 20
component_counts = [2, 3, 4, 5]
X_small = X[:200]

plt.figure(figsize=(7, 4))
for K in component_counts:
    losses = fit_bernoulli_mixture(X_small, n_components=K, n_iter=n_iter, seed=0)["losses"]
    plt.plot(np.arange(1, n_iter + 1), losses, label=f"K={K}")

plt.xlabel("EM iteration")
plt.ylabel("Negative log-likelihood")
plt.title("Effect of the number of mixture components")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()


In [ ]:
sample_sizes = [200, 300, 400, 500]
K = 3

plt.figure(figsize=(7, 4))
for N in sample_sizes:
    losses = fit_bernoulli_mixture(X[:N], n_components=K, n_iter=n_iter, seed=0)["losses"] / N
    plt.plot(np.arange(1, n_iter + 1), losses, label=f"N={N}")

plt.xlabel("EM iteration")
plt.ylabel("NLL per observation")
plt.title("Effect of the number of training examples")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()


In [ ]:
seeds = [0, 1, 2]
K = 3
X_small = X[:200]

plt.figure(figsize=(7, 4))
for seed in seeds:
    losses = fit_bernoulli_mixture(X_small, n_components=K, n_iter=n_iter, seed=seed)["losses"]
    plt.plot(np.arange(1, n_iter + 1), losses, label=f"seed={seed}")

plt.xlabel("EM iteration")
plt.ylabel("Negative log-likelihood")
plt.title("Sensitivity to random initialisation")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.show()


## Sampling from the fitted model

A Bernoulli mixture is a generative model: after training, we can sample a latent component and then
sample pixels independently from that component's Bernoulli probabilities. The optional temperature
transform sharpens uncertain probabilities and makes generated digits visually clearer.


In [ ]:
def sharpen_probabilities(mu, alpha=1.0, eps=1e-6):
    """Sharpen Bernoulli probabilities while keeping them in [0, 1]."""
    mu = np.clip(mu, eps, 1 - eps)
    return (mu**alpha) / (mu**alpha + (1 - mu) ** alpha)


def sample_bernoulli_mixture(pi, mu, n_samples=10, alpha=1.0, seed=42):
    """Generate binary samples from a fitted Bernoulli mixture model."""
    rng = np.random.default_rng(seed)

    pi = np.asarray(pi) / np.sum(pi)
    mu_eff = sharpen_probabilities(np.asarray(mu), alpha=alpha)

    z = rng.choice(len(pi), size=n_samples, p=pi)
    samples = rng.random((n_samples, mu.shape[1])) < mu_eff[z]
    return samples.astype(int), z


In [ ]:
samples, assignments = sample_bernoulli_mixture(
    result["pi"], result["mu"], n_samples=10, alpha=5, seed=42
)
plot_mnist_grid(samples, titles=[f"z={z}" for z in assignments], n_cols=5, figsize=(8, 4))


## Takeaways

- EM gives a monotonic improvement in the negative log-likelihood for this model.
- Log-sum-exp is essential for numerical stability in the E-step.
- Increasing the number of mixture components increases modelling flexibility, but can also increase sensitivity to initialisation.
- The learned component means are interpretable: they act like prototype digit templates.
- Even this simple model can generate recognisable binary digit-like samples, although the independent-pixel assumption makes samples noisy.